# Did the Redesign Help?  Analyzing a UX Experiment

**CS474: Human Computer Interaction — Experimentally Evaluating UX**

Watching users tells you *what* goes wrong; controlled experiments tell you whether your fix *actually helps*.  In this notebook you will analyze a simulated usability study the way a UX researcher would:

1. Simulate task-completion times for an **A/B test** of two checkout designs
2. Visualize the distributions (always look before you test!)
3. Run an **independent-samples t-test** and interpret the p-value
4. Compute an **effect size** (Cohen's d) — because "significant" is not the same as "meaningful"
5. Extend to three designs with a one-way **ANOVA**

We use `numpy`, `scipy`, and `matplotlib` — all pre-installed on Google Colab.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

rng = np.random.default_rng(474)

## Part 1: Simulate the Study

30 participants per condition complete a checkout task.  Design B removes two form fields and adds inline error messages, so we simulate it as genuinely ~6 seconds faster on average.  Human timing data is right-skewed (occasional long struggles), so we use a lognormal distribution rather than a plain normal one.

In [ ]:
n = 30
time_A = rng.lognormal(mean=np.log(48), sigma=0.25, size=n)  # original design
time_B = rng.lognormal(mean=np.log(42), sigma=0.25, size=n)  # redesigned checkout

print(f"Design A: mean {time_A.mean():5.1f} s, median {np.median(time_A):5.1f} s")
print(f"Design B: mean {time_B.mean():5.1f} s, median {np.median(time_B):5.1f} s")

## Part 2: Look at the Data First

Summary statistics hide a lot.  Boxplots (or better, dot plots) show spread, skew, and outliers — a single confused participant can move a mean by seconds.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.boxplot([time_A, time_B], tick_labels=['Design A', 'Design B'])
for i, data in enumerate([time_A, time_B], 1):
    jitter = rng.normal(0, 0.04, len(data))
    ax.plot(np.full(len(data), i) + jitter, data, 'o', alpha=0.4, ms=4)
ax.set_ylabel('task completion time (s)')
ax.set_title('Checkout time by design (each dot = one participant)')
plt.show()

## Part 3: The t-Test

**Null hypothesis:** the two designs have the same mean completion time; any difference we see is sampling luck.  The t-test asks how surprising our observed difference would be if that were true.  We use **Welch's t-test** (`equal_var=False`), which does not assume equal variances and is the safer default.

In [ ]:
t_stat, p_value = stats.ttest_ind(time_A, time_B, equal_var=False)
print(f"t = {t_stat:.2f}, p = {p_value:.4f}")
if p_value < 0.05:
    print("p < 0.05: unlikely under the null; evidence that the designs differ.")
else:
    print("p >= 0.05: this study does NOT provide evidence of a difference.")
    print("(Note: that is not proof the designs are the same!)")

## Part 4: Effect Size — Is the Difference *Meaningful*?

With enough participants, even a 0.1-second difference becomes "significant."  **Cohen's d** expresses the difference in units of standard deviation: ~0.2 is small, ~0.5 medium, ~0.8 large.  Product decisions should weigh effect size (and its real-world meaning: seconds saved, errors avoided) — not just the p-value.

In [ ]:
def cohens_d(a, b):
    pooled_sd = np.sqrt(((len(a) - 1) * a.std(ddof=1) ** 2 +
                         (len(b) - 1) * b.std(ddof=1) ** 2) / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / pooled_sd

d = cohens_d(time_A, time_B)
print(f"Cohen's d = {d:.2f}")
print(f"Design B saves about {time_A.mean() - time_B.mean():.1f} s per checkout on average.")

## Part 5: Three Designs — One-Way ANOVA

Suppose a third variant, C, is also in the test.  Running three pairwise t-tests inflates your false-positive rate; instead, an **ANOVA** first asks "is there *any* difference among the groups?"  If (and only if) it is significant, follow up with pairwise comparisons using a correction (e.g., Bonferroni: divide alpha by the number of comparisons).

In [ ]:
time_C = rng.lognormal(mean=np.log(47), sigma=0.25, size=n)  # C: minor tweak only

f_stat, p_anova = stats.f_oneway(time_A, time_B, time_C)
print(f"ANOVA: F = {f_stat:.2f}, p = {p_anova:.4f}")

if p_anova < 0.05:
    print("\nFollow-up pairwise t-tests (Bonferroni-corrected alpha = 0.05/3 = 0.0167):")
    for name, (x, y) in {'A vs B': (time_A, time_B),
                         'A vs C': (time_A, time_C),
                         'B vs C': (time_B, time_C)}.items():
        t, p = stats.ttest_ind(x, y, equal_var=False)
        print(f"  {name}: p = {p:.4f} {'*' if p < 0.05/3 else ''}")

## Your Turn

1. **Shrink the sample.**  Rerun with `n = 8` per group (a typical quick usability study!).  Does the t-test still detect the difference?  What does this say about small-sample UX claims?
2. **No real difference.**  Set Design B's mean to `np.log(48)` (identical to A) and rerun the whole notebook 5 times with different seeds.  Do you ever get p < 0.05?  What is that phenomenon called?
3. **Beyond time.**  Completion time is only one measure.  Pick two others (error count, SUS satisfaction score, abandonment rate) and describe how each changes what "better design" means.
4. **Design connection.**  In your final project's user experience study, which comparison will you run, what will you measure, and how many participants can you realistically get?  Sketch the analysis *before* collecting data — that's a pre-registration.

## Reflection

Norman's *Design of Everyday Things* Ch. 5 argues that "human error" is usually design error.  Experiments are how we prove a redesign actually removed the error — and how we keep ourselves honest when we're attached to our own designs.